# Train from declared local manifests

Sources → license/path/overlap checks → bounded audio and targets → Trainer evaluation.
GPU fit, convergence and accuracy are unmeasured. Resume selects a valid saved-step marker;
weight and optimizer completeness remains Trainer's runtime check.

In [ ]:
%pip install -q -e '/kaggle/working/shono[audio]'
%pip install -q 'transformers>=4.46.3,<5' 'torch>=2.6,<3' 'accelerate>=1.1,<2' 'librosa>=0.10,<0.12' 'bitsandbytes>=0.43,<0.49'


In [ ]:
import json
from pathlib import Path

from shono.data import LicenseRegistry, Manifest
from shono.protocol import BASE_MODEL_ID, TRAIN_MANIFESTS
from shono.provenance import RunContext, seed_everything
from shono.train import TrainConfig, build_examples, render_experiment, run_training, total_hours

REPO = Path('/kaggle/working/shono')
DATA = REPO / 'data'
AUDIO_ROOT = Path('/kaggle/input')
OUTPUT_DIR = REPO / 'outputs' / 'training'
MODEL_ID = BASE_MODEL_ID
SEED = 42
registry = LicenseRegistry.load(DATA / 'licenses.json')
cfg = TrainConfig(model_id=MODEL_ID, output_dir=str(OUTPUT_DIR), seed=SEED)
seed_everything(SEED)

## Declared train, development and held-out inputs

The default mix is CV, SLR53 and MUCS. Loop is excluded while its permissions are
unconfirmed, and its recording-level transcript is not an aligned training target.
The public training function receives manifests, so detached examples cannot bypass preflight.

In [ ]:
train_manifests = [Manifest.from_jsonl(REPO / path) for path in TRAIN_MANIFESTS]
eval_manifests = [Manifest.from_jsonl(DATA / 'cv-bn-dev.manifest.jsonl')]
held_out_manifests = [Manifest.from_jsonl(DATA / name) for name in (
    'cv-bn-test.manifest.jsonl', 'fleurs-bn-test.manifest.jsonl', 'mucs-cs-test.manifest.jsonl',
)]
selection = dict(chunk_length_s=cfg.chunk_length_s, min_chunk_length_s=cfg.min_chunk_length_s,
                 timestamp_sample_fraction=cfg.timestamp_sample_fraction,
                 seed=cfg.seed, registry=registry)
train_examples = build_examples(train_manifests, **selection)
print(f'{len(train_examples)} train examples, {total_hours(train_examples):.2f} selected h')



REVIEW_PATH = REPO / 'outputs' / 'text-review.json'
text_decisions = json.loads(REVIEW_PATH.read_text()) if REVIEW_PATH.is_file() else {}

## Explicit training run

Preflight validates every declared input before processor or model loading.
Checkpoint/experiment writes use the separate output tree. Final metrics include generated
development WER/CER; held-out report scores require their own complete predictions.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    metrics = run_training(
        cfg, train_manifests, eval_manifests, registry=registry, audio_root=AUDIO_ROOT,
        held_out_manifests=held_out_manifests, text_decisions=text_decisions,
    )
    print(metrics)
else:
    metrics = None
    print('pending: set RUN_TRAINING=True after local inputs and review decisions are ready')

## Experiment record

The existing `full` report specification describes held-out evaluation.
A pending record contains no invented accuracy or completed-training claim.

In [ ]:
ctx = RunContext.capture(cfg.seed, cfg.to_dict(),
                         command='notebooks/train_whisper_medium.ipynb', repo=str(REPO))
record = render_experiment(
    name='whisper-medium-bn-ft-v1', hypothesis='compare the declared fine-tune with the base checkpoint',
    config=cfg, run_context=ctx, data_sources=sorted({m.source for m in train_manifests}),
    n_examples=len(train_examples), train_hours=total_hours(train_examples),
    eval_command='python -m shono.eval --report full', metrics=metrics,
)
print(record)
if RUN_TRAINING:
    (OUTPUT_DIR / 'experiment.md').write_text(record, encoding='utf-8')